<a href="https://colab.research.google.com/github/ankurdev1-drth/Deep_Learning-/blob/main/02_Training_Deep_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [1]:
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import torch

layer = nn.Linear(40, 10)
layer.weight.data *= 6 ** 0.5  # kaiming init (or 3 ** 0.5 for LeCun)
torch.zero_(layer.bias.data)

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [3]:
layer.weight.shape

torch.Size([10, 40])

In [4]:
layer.bias.shape

torch.Size([10])

In [5]:
# another way to for initialization
nn.init.kaiming_uniform_(layer.weight)
nn.init.zeros_(layer.bias)

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

This is clearer and less error prone as compared to the first approach

For applying initialization to the weights of every `nn.Linear` layer in a model the simples option is to write a little function that takes a module, checks whether it's an instance of the `nn.Linear` class, and if so , applies the desired initialization function to its weights. and the function can be then applied using the `apply()` method

In [6]:
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

module = nn.Sequential(
    nn.Linear(40, 10),
    nn.ReLU(),
    nn.Linear(10, 1),
    nn.ReLU()
    )
module.apply(use_he_init)

Sequential(
  (0): Linear(in_features=40, out_features=10, bias=True)
  (1): ReLU()
  (2): Linear(in_features=10, out_features=1, bias=True)
  (3): ReLU()
)

### Leaky ReLU

In [7]:
alpha = 0.2
model = nn.Sequential(
    nn.Linear(50, 40), #0
    nn.LeakyReLU(negative_slope=alpha), #1
    nn.Linear(40, 20), #2
    nn.LeakyReLU(negative_slope=alpha) #3

)
nn.init.kaiming_uniform_(model[0].weight, alpha,
nonlinearity="leaky_relu")
nn.init.kaiming_uniform_(model[2].weight, alpha,nonlinearity="leaky_relu")

Parameter containing:
tensor([[-0.1083, -0.3471, -0.2031,  0.1793, -0.3459,  0.1296, -0.1972, -0.0865,
          0.1692, -0.1971, -0.2943, -0.1412,  0.3445,  0.0068, -0.3728,  0.3564,
          0.3702,  0.2301, -0.2296,  0.3572,  0.3645, -0.1966,  0.2492, -0.3559,
          0.2933,  0.0130,  0.2873,  0.2626,  0.0483,  0.3320,  0.3184, -0.2405,
          0.3614,  0.3220, -0.1219,  0.1351, -0.1851, -0.2677,  0.1465,  0.2378],
        [ 0.3230, -0.1127, -0.2498,  0.2975,  0.2352, -0.1357,  0.2718,  0.2691,
         -0.2408,  0.2551, -0.2360, -0.3019, -0.2420,  0.2699, -0.3709, -0.2094,
          0.1681, -0.0235, -0.1156, -0.0799,  0.0051, -0.0448,  0.2652, -0.2889,
          0.1018, -0.1767, -0.1246,  0.3544, -0.0839,  0.1753, -0.1954, -0.3688,
          0.3090,  0.2706, -0.3388,  0.0024, -0.0583,  0.3532, -0.1259,  0.3092],
        [ 0.2781,  0.2155,  0.2909, -0.1712,  0.2990, -0.2442,  0.0761,  0.2678,
         -0.1252, -0.3235,  0.2856, -0.0768, -0.1562, -0.0289,  0.0670, -0.3290,
    

### Batch Normalization

In [8]:
model = nn.Sequential(
    nn.Flatten(),
    nn.BatchNorm1d(1 * 28 * 28),
    nn.Linear(1 * 28 * 28,  300),
    nn.ReLU(),
    nn.BatchNorm1d(300),
    nn.Linear(300, 100),
    nn.ReLU(),
    nn.BatchNorm1d(100),
    nn.Linear(100, 10)
)

In [9]:
dict(model[1].named_parameters()).keys()

dict_keys(['weight', 'bias'])

In [10]:
dict(model[1].named_buffers()).keys()

dict_keys(['running_mean', 'running_var', 'num_batches_tracked'])

In [11]:
dict(model[1].named_children()).keys()

dict_keys([])

In [12]:
dict(model[1].named_modules()).keys()

dict_keys([''])

Note:
- if BN layers before the activation functions, we can also remove the bias term from the previous `nn.Linear` layers by setting the `bias` hyperparameter to `False`.

In [13]:
Model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(1 * 28 * 28, 300, bias=False),
    nn.BatchNorm1d(300),
    nn.ReLU(),
    nn.Linear(300, 100, bias=False),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

### Layer Normalization

In [14]:
inputs = torch.randn(32, 3, 100, 200)   # batch of random RGB images
layer_norm = nn.LayerNorm([100, 200])
result = layer_norm(inputs)
result

tensor([[[[-7.9590e-01,  8.0980e-01, -1.2385e+00,  ..., -1.6015e-01,
           -2.0287e+00, -1.4864e-01],
          [-3.0187e-01, -9.8044e-01, -1.1252e+00,  ..., -1.2227e+00,
           -8.9815e-02, -1.0295e+00],
          [ 9.2479e-01,  7.0675e-01, -1.3023e+00,  ..., -1.4137e-01,
            6.7732e-01,  1.5780e+00],
          ...,
          [ 4.1848e-01,  3.1396e-01,  1.3668e+00,  ...,  7.0603e-01,
            3.4129e-01, -1.2566e-01],
          [-1.0192e+00,  5.8226e-01, -1.6953e-01,  ...,  1.1465e+00,
            1.3503e+00,  1.4185e+00],
          [-1.7254e+00,  1.6202e+00,  7.1738e-01,  ...,  2.4339e-01,
           -1.3836e+00, -4.0555e-01]],

         [[-1.1327e+00, -1.6254e+00, -6.7016e-01,  ..., -4.4840e-01,
            9.2056e-01, -7.8154e-01],
          [-5.4309e-01, -7.6435e-01, -2.0410e-01,  ..., -2.3013e-01,
            1.0598e+00,  6.2163e-01],
          [-1.6567e-01, -1.1805e+00,  7.1146e-01,  ...,  4.7304e-01,
           -4.3590e-01,  6.1376e-02],
          ...,
     

In [15]:
# method 2 to perform the same thing as above!
means = inputs.mean(dim=[2,3], keepdim=True)  #shape [32, 3, 1, 1]
vars_ = inputs.var(dim=[2,3], keepdim=True, unbiased=False) # shape: same
stds = torch.sqrt(vars_ + layer_norm.eps) # eps is a smoothing term (1e-5)
result = layer_norm.weight * (inputs - means) / stds + layer_norm.bias

In [16]:
layer_norm = nn.LayerNorm([3, 100, 200])
result = layer_norm(inputs)

In [17]:
result

tensor([[[[-7.9106e-01,  8.1700e-01, -1.2343e+00,  ..., -1.5438e-01,
           -2.0257e+00, -1.4285e-01],
          [-2.9631e-01, -9.7588e-01, -1.1208e+00,  ..., -1.2185e+00,
           -8.3940e-02, -1.0250e+00],
          [ 9.3216e-01,  7.1380e-01, -1.2982e+00,  ..., -1.3557e-01,
            6.8433e-01,  1.5863e+00],
          ...,
          [ 4.2510e-01,  3.2043e-01,  1.3748e+00,  ...,  7.1308e-01,
            3.4779e-01, -1.1984e-01],
          [-1.0147e+00,  5.8912e-01, -1.6378e-01,  ...,  1.1542e+00,
            1.3583e+00,  1.4266e+00],
          [-1.7219e+00,  1.6286e+00,  7.2444e-01,  ...,  2.4975e-01,
           -1.3796e+00, -4.0014e-01]],

         [[-1.1334e+00, -1.6272e+00, -6.6980e-01,  ..., -4.4751e-01,
            9.2465e-01, -7.8144e-01],
          [-5.4243e-01, -7.6421e-01, -2.0265e-01,  ..., -2.2874e-01,
            1.0642e+00,  6.2502e-01],
          [-1.6412e-01, -1.1813e+00,  7.1506e-01,  ...,  4.7608e-01,
           -4.3499e-01,  6.3453e-02],
          ...,
     

## Gradient Clipping

See the line nn.utils.clip_grad_norm_(...) in the training function in the next section.

# Reusing Pretrained Layers

### Transfer Learning with PyTorch

- X_train_A = images of all items except for T-shirts/tops and pullovers (classes 0 and 2 )
- X_train_B = first 20 images of Tshirt and Pullovers

The validation set and the test set are also split this way, but without restricting the number of images.

We will train a model on set A (classification task with 8 classes), and try to reuse it to tackle set B (binary classification). We hope to transfer a little bit of knowledge from task A to task B, since classes in set A (trousers, dresses, coats, sandals, shirts, sneakers, bags, and ankle boots) are somewhat similar to classes in set B (T-shirts/tops and pullovers). However, since we are using Linear layers, only patterns that occur at the same location can be reused (in contrast, convolutional layers will transfer much better, since learned patterns can be detected anywhere on the image)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_openml


fashion_mnist = fetch_openml(name="Fashion-MNIST", as_frame=False) #as_frames = False for getting the numpy arrays instead of pandas dataframe/series
X = torch.FloatTensor(fashion_mnist.data.reshape(-1, 1, 28, 28) / 255.)
y = torch.from_numpy(fashion_mnist.target.astype(int))
in_B = (y == 0) | (y == 2)  # Pullover or T-shirt/top
X_A, y_A = X[~in_B], y[~in_B]
y_A = torch.maximum(y_A - 2, torch.tensor(0))  # [1,3,4,5,6,7,8,9] => [0,..,7]
X_B, y_B = X[in_B], (y[in_B] == 2).to(dtype=torch.float32).view(-1, 1)

train_set_A = TensorDataset(X_A[:-7_000], y_A[:-7000])
valid_set_A = TensorDataset(X_A[-7_000:-5000], y_A[-7000:-5000])
test_set_A = TensorDataset(X_A[-5_000:], y_A[-5000:])
train_set_B = TensorDataset(X_B[:20], y_B[:20])
valid_set_B = TensorDataset(X_B[20:5000], y_B[20:5000])
test_set_B = TensorDataset(X_B[5_000:], y_B[5000:])

train_loader_A = DataLoader(train_set_A, batch_size=32, shuffle=True)
valid_loader_A = DataLoader(valid_set_A, batch_size=32)
test_loader_A = DataLoader(test_set_A, batch_size=32)
train_loader_B = DataLoader(train_set_B, batch_size=32, shuffle=True)
valid_loader_B = DataLoader(valid_set_B, batch_size=32)
test_loader_B = DataLoader(test_set_B, batch_size=32)

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device


In [ ]:
%pip install torchmetrics


import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            # Uncomment to activate gradient clipping:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history



In [ ]:
torch.manual_seed(42)

model_A = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 8)
  )
model_A = model_A.to(device)

In [ ]:
model_A.apply(use_he_init)

In [ ]:
n_epochs = 20
optimizer = torch.optim.SGD(model_A.parameters(), lr=0.005)
loss_fn = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=8).to(device)
history_A = train(model_A, optimizer, loss_fn, accuracy, train_loader_A, valid_loader_A, n_epochs )

In [ ]:
torch.manual_seed(42)
model_B = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.ReLU(),
    nn.Linear(100, 1)
).to(device)

In [ ]:
model_B.apply(use_he_init)

In [ ]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B.parameters(), lr=0.001)
loss_fn = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B, optimizer, loss_fn, accuracy, train_loader_B, valid_loader_B, n_epochs)

In [ ]:
# reusing all the layers except the output layer
import copy

torch.manual_seed(42)
reused_layer = copy.deepcopy(model_A[: -1])
model_B_on_A = nn.Sequential(
    *reused_layer,
    nn.Linear(100, 1) # new output layer for task B
).to(device)

Note:
- `copy.deepcopy()` function to copy all the modules and submodules in `nn.Sequential` module
- model_B_on_A is another `nn.Sequential` module based on the reused layers of model A plus a new output layer for task B it has a single output since task B is binary classification.

In [ ]:
# freezing the reused layers during the first few epochs:
for layer in model_B_on_A[:-1]:
  for param in layer.parameters():
    param.requires_grad = False


In [ ]:
n_epochs = 10
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.005)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                  train_loader_B, valid_loader_B, n_epochs)

In [ ]:
# unfreezing the reused layers during the first few epochs:
for layer in model_B_on_A[2:]:
  for param in layer.parameters():
    param.requires_grad = True

In [ ]:
n_epochs = 20
optimizer = torch.optim.SGD(model_B_on_A.parameters(), lr=0.01)
xentropy = nn.BCEWithLogitsLoss()
accuracy = torchmetrics.Accuracy(task="binary").to(device)
history_B = train(model_B_on_A, optimizer, xentropy, accuracy,
                train_loader_B, valid_loader_B, n_epochs)

In [ ]:
evaluate_tm(model_B_on_A, test_loader_B, accuracy)

## Faster Optimizers

In [ ]:

def build_model(seed=43):
  torch.manual_seed(seed)
  model = nn.Sequential(
      nn.Flatten(),
      nn.Linear(28 * 28, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 100),
      nn.ReLU(),
      nn.Linear(100, 10)
  ).to(device)
  model.apply(use_he_init)
  return model

def test_optimizer(model, optimizer, n_epochs=10, batch_size=32):
  train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
  valid_loader = DataLoader(valid_set, batch_size=batch_size)
  test_loader = DataLoader(test_set, batch_size=32)
  xentropy = nn.CrossEntropyLoss()
  accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10)
  history = train(model, optimizer, xentropy, accuracy.to(device),
                  train_loader, valid_loader, n_epochs)
  return history, evaluate_tm(model, test_loader, accuracy)


In [ ]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

## Momentum Optimization

In [ ]:
train_set = TensorDataset(X[:55_000], y[:55_000])
valid_set = TensorDataset(X[55_000:60_000], y[55_000:60_000])
test_set = TensorDataset(X[60_000:], y[60_000:])

model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, lr=0.01)
history_momentum, accuracy_momentum = test_optimizer(model, optimizer)

### Nesterov Accelerated Gradient

In [ ]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, nesterov=True, lr=0.01)
history_nesterov, accuracy_nesterov = test_optimizer(model, optimizer)

## AdaGrad

In [ ]:
model = build_model()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)
history_adagrad, accuracy_adagrad = test_optimizer(model, optimizer)

## RMSProp

In [ ]:
model = build_model()
optimizer = torch.optim.RMSprop(model.parameters(), lr=0.01, alpha=0.9)
history_rmsprop, accuracy_rmsprop = test_optimizer(model, optimizer)

## Adam Optimization

In [ ]:
model = build_model()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05, betas=(0.9, 0.999))
history_adam, accuracy_adam = test_optimizer(model, optimizer)


### Adamax Optimization

In [ ]:
model =  build_model()
optimizer = torch.optim.Adamax(model.parameters(), lr=0.03, betas=(0.9, 0.999))
history_adamax, accuracy_adamax = test_optimizer(model, optimizer)

### Nadam Optimization

In [ ]:
model = build_model()  # extra code
optimizer = torch.optim.NAdam(model.parameters(), betas=(0.9, 0.999), lr=0.05)
history_nadam, acc_nadam = test_optimizer(model, optimizer)

### AdamW Optimization

In [ ]:
model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), betas=(0.9, 0.999), lr=0.03)
history_adamW, accuracy_adamw = test_optimizer(model, optimizer)

In [ ]:
for plot in ("train_losses", "valid_metrics"):
  plt.figure(figsize=(8,4))
  opt_names = "SGD Momentum Nesterov Adagrad RMSProp Adam Adamax Nadam AdamW".split()
  for history, opt_name in zip(
      (history_momentum, history_nesterov, history_adagrad,
       history_rmsprop, history_adam, history_adamax, history_nadam,
       history_adamW), opt_names):
       plt.plot(history[plot], label=opt_name, linewidth=3)

  plt.grid()
  plt.xlabel("Epochs")
  plt.ylabel({"train_losses": "Training_loss", "valid_metrics": "Validataion accuracy"}[plot])
  plt.legend(loc="upper right")
  plt.show()

## Learning Rate Scheduling

In [ ]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
exp_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)

In [ ]:
def train_with_scheduler(model, optimizer, loss_fn, metric, train_loader,
                         valid_loader, n_epochs, scheduler):
  history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
  for epoch in range(n_epochs):
    losses = []
    metric.reset()
    for X_batch, y_batch in train_loader:
      model.train()
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      losses.append(loss.item())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)
  history["train_losses"].append(np.mean(losses))
  history["train_metrics"].append(metric.compute().itemO())
  val_metric = evaluate_tm(model, valid_loader, metric).item()
  history["valid_metrics"].append(val_metric)
  print(f"Epoch{epoch + 1} / {n_epochs},"
        f"train loss : {history['train_losses'][-1]:.4f},"
        f"train metric: {history['train_metrics'][-1]:.4f}, "
        f"valid metric: {history['valid_metrics'][-1]:.4f}")
  if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
    scheduler.step(val_metric)
  else:
    scheduler.step()
  print(f"Learning rate: {scheduler.get_last_lr()[0]:.5f}")
  return history


In [ ]:
def train_with_scheduler(model, optimizer, loss_fn, metric, train_loader,
                         valid_loader, n_epochs, scheduler):
  history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
  for epoch in range(n_epochs):
    losses = []
    metric.reset()
    for X_batch, y_batch in train_loader:
      model.train()
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      losses.append(loss.item())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)

    # Note: These should be inside the epoch loop to track progress
    history["train_losses"].append(np.mean(losses))
    history["train_metrics"].append(metric.compute().item())
    val_metric = evaluate_tm(model, valid_loader, metric).item()
    history["valid_metrics"].append(val_metric)

    print(f"Epoch {epoch + 1} / {n_epochs}, "
          f"train loss : {history['train_losses'][-1]:.4f}, "
          f"train metric: {history['train_metrics'][-1]:.4f}, "
          f"valid metric: {history['valid_metrics'][-1]:.4f}")

    if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
      scheduler.step(val_metric)
    else:
      scheduler.step()

    # Use get_last_lr() for standard schedulers
    if not isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
        print(f"Learning rate: {scheduler.get_last_lr()[0]:.5f}")

  return history

In [ ]:
history_exp, acc_exp = test_scheduler(model, optimizer, exp_scheduler)

In [ ]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
perf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.1
)

In [ ]:
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=32)
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

history_perf = train_with_scheduler(
    model, optimizer, xentropy, accuracy, train_loader,
    valid_loader, n_epochs=10, scheduler=perf_scheduler
)

In [ ]:
model = build_model()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=3
)


In [ ]:
# using lambda to wrap around the customly created function
model = build_model()
optimizer = torch.optim.Adam(model.parameters(), betas=(0.9, 0.999), lr=0.05)
warmup_scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda epoch: (min(epoch, 3) /3) * (1.0 - 0.1) + 0.1
)

In [ ]:
def train_with_warmup(model, optimizer, loss_fn, metric, train_loader,
                      valid_loader, n_epochs, warmup_scheduler, scheduler):
  history = {"train_losses":[], "train_mmetrics": [], "valid_metrics": []}
  for epoch in range(n_epochs):

    warmup_scheduler.step()

    losses = []
    metric.reset()

    for X_batch, y_batch in train_loader:

      model.train()
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      losses.append(loss.item())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
      metric.update(y_pred, y_batch)
    history["train_losses"].append(np.mean(losses))
    history["train_metrics"].append(metric.compute().item())
    val_metric = evaluate_tm(model, valid_loader, metric).item()
    history["valid_metrics"].append(val_metric)
    print(f"Epoch {epoch + 1 }/ {n_epochs},"
          f"train loss: {history['train_losses'][-1]:.4f},"
          f"valid metric: {history['valid_metrics'][-1]:.4f}")

    if epoch >=3:
      scheduler.step(val_metric) # deactivate other scheduler(s) during warmup

    # exttra code
    print(f"Learning rate : {scheduler.get_last_lr()[0]:.5f}")
  return history

def test_warmup_scheduler(model, optimizer, warmup_scheduler, scheduler,
                          n_epochs=10, batch_size=32):
  train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
  valid_loader = DataLoader(valid_set, batch_size=32)
  test_loader = DataLoader(test_set, batch_size=32)
  xentropy = nn.CrossEntropyLoss()
  accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
  history = train_with_warmup(
      model, optimizer, xentropy, accuracy, train_loader,
      valid_loader, n_epochs, warmup_scheduler, scheduler
  )
  return history, evaluate_tm(model, test_loader, metric)